# The transformation logic

In [0]:
%python
query = """
SELECT
    ROW_NUMBER() OVER (ORDER BY ci.customer_id) AS customer_key,
    ci.customer_id,
    ci.customer_number,
    ci.first_name,
    ci.last_name,
    ci.marital_status,
    CASE
        WHEN ci.gender <> 'n/a' THEN ci.gender
        ELSE COALESCE(ca.gender, 'n/a')
    END AS gender,
    ci.created_date AS create_date,
    lo.country,
    ca.birth_date AS birthdate
FROM silver.crm_customers ci
LEFT JOIN silver.erp_customer_location lo
    ON ci.customer_number = lo.customer_number
LEFT JOIN silver.erp_customers ca
    ON ci.customer_number = ca.customer_number
"""

df = spark.sql(query)
df.limit(10).display()

# Writing gold table

In [0]:
%python
df.write.mode("overwrite").format('delta').saveAsTable("workspace.gold.dim_customers")

# Sanity checks of gold table

In [0]:
SELECT * FROM gold.dim_customers LIMIT 10